In [1]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
import torch

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

In [2]:
from torch.utils.data import TensorDataset,  DataLoader
import torch.nn as nn

In [3]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

In [4]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)


In [5]:
train_data_wd = TensorDataset(X_train[:, :4], X_train[:, 4:], y_train)
valid_data_wd = TensorDataset(X_valid[:, :4], X_valid[:, 4:], y_valid)
test_data_wd = TensorDataset(X_test[:, 4:], X_test[:, 4:], y_test)

In [6]:
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
valid_loader_wd  =DataLoader(valid_data_wd, batch_size=32, shuffle=True)
test_data_wd = DataLoader(test_data_wd, batch_size=32, shuffle=True)

In [7]:
class DiabetesWideAndDeep(nn.Module):
    def __init__(self, n_features_wide, n_features_deep):
        super().__init__()
        self.deepstack = nn.Sequential(
            nn.Linear(n_features_deep, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU()
        )
        self.output_layer = nn.Linear(n_features_wide +10, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deepstack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)


In [8]:
device = 'cuda'

In [13]:
model = DiabetesWideAndDeep(n_features_wide=4, n_features_deep=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [14]:
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for X_batch_wide, X_batch_deep, y_batch in train_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch_wide, X_batch_deep)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)

        model.eval()
        total_valid_loss = 0.0

        with torch.no_grad():
            for X_wide, X_deep, y_batch, in valid_loader:
                X_wide = X_wide.to(device)
                X_deep = X_deep.to(device)
                y_batch = y_batch.to(device)

                y_pred = model(X_wide, X_deep)
                loss = criterion(y_pred, y_batch)
                total_valid_loss += loss.item()
        mean_valid_loss = total_valid_loss / len(valid_loader)
        print(f"Epoch {epoch + 1:02d}/{n_epochs} | Train Loss: {mean_loss:.2f} | Valid Loss: {mean_valid_loss:.2f}")

In [16]:
train(model, optimizer, criterion, train_loader_wd, valid_loader_wd, n_epochs=100)

Epoch 01/100 | Train Loss: 18944.76 | Valid Loss: 20904.98
Epoch 02/100 | Train Loss: 16830.06 | Valid Loss: 19954.57
Epoch 03/100 | Train Loss: 17970.63 | Valid Loss: 19821.35
Epoch 04/100 | Train Loss: 16177.65 | Valid Loss: 18875.33
Epoch 05/100 | Train Loss: 17124.17 | Valid Loss: 18645.91
Epoch 06/100 | Train Loss: 15582.94 | Valid Loss: 17706.15
Epoch 07/100 | Train Loss: 14401.26 | Valid Loss: 16970.22
Epoch 08/100 | Train Loss: 14103.47 | Valid Loss: 16296.66
Epoch 09/100 | Train Loss: 14459.06 | Valid Loss: 15400.99
Epoch 10/100 | Train Loss: 13414.90 | Valid Loss: 14736.13
Epoch 11/100 | Train Loss: 12674.94 | Valid Loss: 14519.88
Epoch 12/100 | Train Loss: 12347.74 | Valid Loss: 13669.77
Epoch 13/100 | Train Loss: 11486.29 | Valid Loss: 13110.34
Epoch 14/100 | Train Loss: 11109.38 | Valid Loss: 12427.47
Epoch 15/100 | Train Loss: 11424.71 | Valid Loss: 12118.91
Epoch 16/100 | Train Loss: 9764.18 | Valid Loss: 11313.42
Epoch 17/100 | Train Loss: 9632.64 | Valid Loss: 10786.92